# Process Annotated Data for NER & RE

Translatess Label Studio JSON files to a standard format for downstream training and evaluation. 

Performs some exploratory analysis and combines low-prevalence categories.

In [ ]:
import json
import ast
from datetime import datetime
from collections import defaultdict, Counter
from typing import Any, Dict
from dataclasses import dataclass
from itertools import combinations
import random
import copy

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

import spacy
from spacy.tokens import Span
from spacy import displacy


In [ ]:
# Local imports
import sys
from pathlib import Path

# NOTE: A bit of a hack to get relative paths to work
sys.path.append(str(Path.cwd().parent.parent / "src"))
from common.paths import RAW, PROCESSED, PREDICTIONS
from common.graph_helpers import analyze_category_distribution_from_dict, plot_cooccurrence_heatmap
from common.logging import setup_logger
from common.json_helpers import save_json
from shared.evaluators import SpanEvaluator


In [ ]:
@dataclass
class Config:
    logger = setup_logger('preprocessing.process_annotated_data')
    annotated_data_path = RAW / "annotations" / "annotated-gold-standard.json"
    taxonomy_path = PROCESSED / "taxonomy_autogen_v2.csv"
    output_path = PROCESSED / "gold_standard.json"
    
    random_state = 42
    person_labels = {"Person_Name", "Person_Role", "Person_Pronoun"}
    cat_labels = []

config = Config()


 ## 1. Import Data

In [ ]:
with open(config.annotated_data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Total raw notes loaded: {len(data)}")

taxonomy = pd.read_csv(config.taxonomy_path)
config.cat_labels = set(taxonomy.cat_label.values.tolist())
taxonomy.head(5)


 ## 2. Formatting & Extraction

 Strip out Label Studio Metadata, ensure spans are aligned with word boundaries.

In [ ]:
def clean_span(text: str, start: int, end: int):
    '''Validates and cleans Label Studio character offsets.'''
    if start < 0 or end > len(text) or start >= end:
        return None

    # Expand boundaries if they split a word
    while start > 0 and text[start - 1].isalnum(): start -= 1
    while end < len(text) and text[end].isalnum(): end += 1

    span_text = text[start:end]
    cleaned = span_text.strip()
    if not cleaned: return None

    leading_whitespace = len(span_text) - len(span_text.lstrip())
    new_start = start + leading_whitespace
    
    return {"text": cleaned, "start": new_start, "end": new_start + len(cleaned)}

def clean_relation(from_id: str, to_id: str, from_label: str, to_label: str) -> dict | None:
    '''Ensures relations are formatted correctly (Need -> Person).'''
    is_from_person = from_label in config.person_labels
    is_to_person = to_label in config.person_labels

    if not is_from_person and is_to_person:
        return {"from": from_id, "to": to_id}
    elif is_from_person and not is_to_person:
        return {"from": to_id, "to": from_id}
    else:
        return None

def extract_annotations(task: Dict[str, Any]) -> Dict[str, Any] | None:
    '''
    Extract annotations from Label Studio.
    Output schema: {id: uuid, text: str, needs: list(dict), persons: list(dict), relations: list(dict)}
    '''
    id = task["data"]["id"]
    text = task["data"]["note_content"]
    
    if not task["annotations"]:
        return None
        
    annotation = min(task["annotations"], key=lambda x: datetime.strptime(x["created_at"], "%Y-%m-%dT%H:%M:%S.%fZ"))
    annotation_date = datetime.strptime(annotation["created_at"], "%Y-%m-%dT%H:%M:%S.%fZ")
    
    needs, persons, relations = [], [], []
    ids_to_labels = {}

    # 1. Extract all entities and labels
    for l in annotation["result"]:
        if l["type"] != "labels": continue
        
        ent_id = l["id"]
        label = l["value"]["labels"][0]
        span = clean_span(text, l["value"]["start"], l["value"]["end"])
        
        if not span: continue
        
        ids_to_labels[ent_id] = label
        item = {"id": ent_id, "text": span["text"], "start": span["start"], "end": span["end"], "label": label}

        if l["from_name"] == "need_labels" and label in config.cat_labels:
            needs.append(item)
        elif l["from_name"] == "entity_labels" and label in config.person_labels:
            persons.append(item)

    # 2. Extract all relations
    for l in annotation["result"]:
        if l["type"] == "relation" and ids_to_labels.get(l["from_id"]) and ids_to_labels.get(l["to_id"]):
            rel = clean_relation(l["from_id"], l["to_id"], ids_to_labels[l["from_id"]], ids_to_labels[l["to_id"]])
            if rel:
                relations.append(rel)

    return {
        "id": id,
        "label_studio_id": task['id'],
        "annotation_date": annotation_date,
        "text": text,
        "needs": needs,
        "persons": persons,
        "relations": relations,
    }

records = [r for r in (extract_annotations(task) for task in data) if r is not None]
records[1]


 ## 3. EDA

 ### Relation Distances (proportion of full note)

In [ ]:
relative_distances = []

for r in records:
    # 1. Determine Document Length
    # Try to get the actual text length first
    doc_text = r.get('text', '')
    doc_length = len(doc_text)
    
    # Skip if length is 0 to avoid division by zero errors
    if doc_length == 0:
        continue
    
    all_ents = r.get('needs') + r.get('persons')
    ents_by_id = {e['id']: e for e in all_ents}
    
    # 2. Calculate Relative Distances
    for rel in r['relations']:
        from_ent = ents_by_id.get(rel['from'])
        to_ent = ents_by_id.get(rel['to'])
        
        if from_ent and to_ent:
            # Absolute distance (0 if overlapping)
            abs_dist = max(0, max(from_ent['start'], to_ent['start']) - min(from_ent['end'], to_ent['end']))
            
            # Relative distance (Percentage of total document length)
            rel_dist = (abs_dist / doc_length) * 100
            relative_distances.append(rel_dist)

if relative_distances:
    # PLOT 1: Relative Distance Histogram
    plt.figure(figsize=(10, 5), dpi=100)
    sns.histplot(relative_distances, bins=25, kde=True)
    
    plt.title('Relative Character Distance Between Related Entities')
    plt.xlabel('Distance as Percentage of Document Length (%)')
    plt.ylabel('Frequency')
    plt.xlim(0, 100)
    
    plt.tight_layout()
    plt.show()

In [ ]:
print(np.percentile(relative_distances, 99))
np.histogram([r for r in relative_distances if r < 75], 10)

### See which classes overlap most

In [ ]:
parent_mapping = pd.Series(taxonomy.high_level_category.values,index=taxonomy.cat_label).to_dict()

records_parents = [
    {**r, 'needs': [{**n, 'label': parent_mapping.get(n['label'], n['label'])} for n in r['needs']]}
    for r in records
]


In [ ]:
# 1. Track overlapping pairs
overlap_pairs_counts = Counter()
docs_with_overlaps = 0

total_entities = 0
for record in records:
    entities = record.get("needs", []) + record.get("persons", [])
    has_overlap = False
    total_entities += len(entities)
    
    for e1, e2 in combinations(entities, 2):
        if max(e1["start"], e2["start"]) < min(e1["end"], e2["end"]):
            has_overlap = True
            lbl_pair = tuple(sorted([e1["label"], e2["label"]]))
            overlap_pairs_counts[lbl_pair] += 1
            
    docs_with_overlaps += has_overlap

total_pairs = sum(overlap_pairs_counts.values())


# 2. Render Matrix and Heatmap (if overlaps exist)
if overlap_pairs_counts:
    labels = sorted({lbl for pair in overlap_pairs_counts for lbl in pair})
    overlap_matrix_df = pd.DataFrame(0, index=labels, columns=labels)

    # Populate symmetric matrix directly
    for (l1, l2), count in overlap_pairs_counts.items():
        overlap_matrix_df.loc[l1, l2] = overlap_matrix_df.loc[l2, l1] = count

    # Plot
    n = len(labels)
    plt.figure(figsize=(max(10, n * 0.45), max(8, n * 0.35)))
    sns.set_theme(style="white")
    
    mask = np.triu(np.ones(overlap_matrix_df.shape, dtype=bool), k=1)
    sns.heatmap(
        overlap_matrix_df, mask=mask, annot=True, fmt="d", cmap="YlOrRd", 
        cbar_kws={'label': 'Collision Count'}, linewidths=0.5, linecolor="#eeeeee"
    )
    
    plt.title("Matrix of Span Overlaps", fontsize=14, pad=20, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

print(f"Documents with overlaps: {docs_with_overlaps}/{len(records)} ({docs_with_overlaps/len(records)*100:.1f}%)")
print(f"Total entities: {total_entities} | Overlapping pairs: {total_pairs} ({total_pairs/total_entities*100:.1f}%)")

In [ ]:
# 1. Track overlapping pairs
overlap_pairs_counts = Counter()
docs_with_overlaps = 0

total_entities = 0
for record in records_parents:
    entities = record.get("needs", []) + record.get("persons", [])
    has_overlap = False
    total_entities += len(entities)
    
    for e1, e2 in combinations(entities, 2):
        if max(e1["start"], e2["start"]) < min(e1["end"], e2["end"]):
            has_overlap = True
            lbl_pair = tuple(sorted([e1["label"], e2["label"]]))
            overlap_pairs_counts[lbl_pair] += 1
            
    docs_with_overlaps += has_overlap

total_pairs = sum(overlap_pairs_counts.values())


# 2. Render Matrix and Heatmap (if overlaps exist)
if overlap_pairs_counts:
    labels = sorted({lbl for pair in overlap_pairs_counts for lbl in pair})
    overlap_matrix_df = pd.DataFrame(0, index=labels, columns=labels)

    # Populate symmetric matrix directly
    for (l1, l2), count in overlap_pairs_counts.items():
        overlap_matrix_df.loc[l1, l2] = overlap_matrix_df.loc[l2, l1] = count

    # Plot
    n = len(labels)
    plt.figure(figsize=(max(10, n * 0.45), max(8, n * 0.35)))
    sns.set_theme(style="white")
    
    mask = np.triu(np.ones(overlap_matrix_df.shape, dtype=bool), k=1)
    sns.heatmap(
        overlap_matrix_df, mask=mask, annot=True, fmt="d", cmap="YlOrRd", 
        cbar_kws={'label': 'Collision Count'}, linewidths=0.5, linecolor="#eeeeee"
    )
    
    plt.title("Matrix of Span Overlaps", fontsize=14, pad=20, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

print(f"Documents with overlaps: {docs_with_overlaps}/{len(records)} ({docs_with_overlaps/len(records)*100:.1f}%)")
print(f"Total entities: {total_entities} | Overlapping pairs: {total_pairs} ({total_pairs/total_entities*100:.1f}%)")

In [ ]:
# Track single span multi label pairs
ml_pairs_counts = Counter()
docs_with_mls = 0

total_entities = 0
for record in records:
    entities = record.get("needs", []) + record.get("persons", [])
    is_multi_label = False
    total_entities += len(entities)
    
    for e1, e2 in combinations(entities, 2):
        if e1["start"] == e2["start"] and e1["end"] == e2["end"]:
            is_multi_label = True
            lbl_pair = tuple(sorted([e1["label"], e2["label"]]))
            ml_pairs_counts[lbl_pair] += 1
            
    docs_with_mls += is_multi_label

total_pairs = sum(ml_pairs_counts.values())


# 2. Render Matrix and Heatmap (if multi label spans exist)
if ml_pairs_counts:
    labels = sorted({lbl for pair in ml_pairs_counts for lbl in pair})
    multi_label_matrix_df = pd.DataFrame(0, index=labels, columns=labels)

    # Populate symmetric matrix directly
    for (l1, l2), count in ml_pairs_counts.items():
        multi_label_matrix_df.loc[l1, l2] = multi_label_matrix_df.loc[l2, l1] = count

    # Plot
    n = len(labels)
    plt.figure(figsize=(max(10, n * 0.45), max(8, n * 0.35)))
    sns.set_theme(style="white")
    
    mask = np.triu(np.ones(multi_label_matrix_df.shape, dtype=bool), k=1)
    sns.heatmap(
        multi_label_matrix_df, mask=mask, annot=True, fmt="d", cmap="YlOrRd", 
        cbar_kws={'label': 'Collision Count'}, linewidths=0.5, linecolor="#eeeeee"
    )
    
    plt.title("Matrix of Multi Label Spans", fontsize=14, pad=20, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

print(f"Documents with multi-label spans: {docs_with_mls}/{len(records)} ({docs_with_mls/len(records)*100:.1f}%)")
print(f"Total entities: {total_entities} | Multi-label spans: {total_pairs} ({total_pairs/total_entities*100:.1f}%)")

### Analyse Spans

In [ ]:
# Set up a 2x2 grid for our plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------
# 1. Person Label Counts
# ---------------------------------------------------------
person_types_in_rel = [p['label'] for rec in records for r in rec['relations'] for p in rec.get('persons', []) if p['id'] in r.values()]
sns.countplot(x=person_types_in_rel, ax=axes[0, 0], palette="Blues_r")
axes[0, 0].set_title("Person Labels Distribution (with relations)", fontweight='bold')
axes[0, 0].set_xlabel("Count")
axes[0, 0].set_ylabel("")

# ---------------------------------------------------------
# 2. Relation Counts per Document
# ---------------------------------------------------------
relation_counts = [len(r.get("relations", [])) for r in records]
sns.countplot(x=relation_counts, ax=axes[0, 1], palette="magma")
axes[0, 1].set_title("Relation Counts per Document", fontweight='bold')
axes[0, 1].set_xlabel("Number of Relations in Doc")
axes[0, 1].set_ylabel("Document Count")

# ---------------------------------------------------------
# 3. Note Lengths (in Words)
# ---------------------------------------------------------
word_counts = [len(r["text"].split()) for r in records]
sns.histplot(word_counts, bins=12, ax=axes[1, 0], color="teal", kde=True)
axes[1, 0].axvline(np.percentile(word_counts, 95), color='red', linestyle='--', label='95th Percentile')
axes[1, 0].set_title("3. Note Lengths (Words)", fontweight='bold')
axes[1, 0].set_xlabel("Word Count")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].legend()

# ---------------------------------------------------------
# 4. Span Lengths (Words for all entities)
# ---------------------------------------------------------
span_lengths = [len(ent["text"].split()) for r in records for ent in r.get("needs", []) + r.get("persons", [])]
sns.histplot(span_lengths, bins=20, ax=axes[1, 1], color="coral", discrete=True)
axes[1, 1].set_title("4. Span Length Distribution (Words)", fontweight='bold')
axes[1, 1].set_xlabel("Word Count")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout(pad=3.0)
plt.show()

# Print out the text statistics for quick reference
print("=" * 60)
print("TEXT STATS SUMMARY")
print("=" * 60)
print(f"Note Word Count -> Median: {np.median(word_counts):.0f} | 95th Percentile: {np.percentile(word_counts, 95):.0f} | Max: {max(word_counts)}")
print(f"Span Word Count -> Median: {np.median(span_lengths):.0f} | 95th Percentile: {np.percentile(span_lengths, 95):.0f} | Max: {max(span_lengths)}")
print("=" * 60)

In [ ]:
pd.Series(person_types_in_rel).value_counts(normalize=True)

In [ ]:
note_lengths_words_raw = [len(r["text"].split()) for r in records]
note_length_words = np.array([l for l in note_lengths_words_raw if l <= 201])

counts, bins = np.histogram(note_length_words, bins=12) # Bin size of 10

# Mean + SD + 95th percentile
mean_val = note_length_words.mean()
standard_dev = note_length_words.std()
p = np.percentile(note_length_words, 95)

print(f"Mean: {mean_val}, Standard Deviation: {standard_dev}, 95th Percentile: {p}")
print(pd.DataFrame(zip(bins, counts), columns=['bin', 'count']))

In [ ]:
(len([c for c in note_lengths_words_raw if c > 256]) / len(note_lengths_words_raw)) * 100

In [ ]:
filtered_lengths = [l for l in span_lengths if l <= 15]
unique_words, counts = np.unique(filtered_lengths, return_counts=True)

for x, y in zip(unique_words, counts):
    print(f"({x}, {y})")

In [ ]:
# Percentage of entity pairs that have a relation
rel_entities_count = [
    (len(r.get("relations", [])), len(r.get("needs", [])), len(r.get("persons", [])))
    for r in records
]

rel_entities_perc = [(r/(n*p)) for (r, n, p) in rel_entities_count if n*p != 0]

sns.histplot(rel_entities_perc, bins=5)
plt.title("Distribution of need/person pairs that have a relation (%)")

np.mean(rel_entities_perc)

In [ ]:
np.histogram(rel_entities_perc, bins=5)

In [ ]:
note_span_counts = [len(r['needs']) + len(r['persons']) for r in records]
ax = sns.histplot(note_span_counts, bins=10)
ax.set_title("Number of spans in notes")
ax.set_ylabel("Number of spans")


np.average(note_span_counts)

### Person Role Examples

In [ ]:
person_roles = [p['text'].lower() for r in records for p in r.get('persons', []) if p['label'] == 'Person_Role']
person_roles_df = pd.Series(person_roles).value_counts().rename_axis('role').reset_index(name='count')
top_roles = person_roles_df[person_roles_df['count'] > 10]

plt.figure(figsize=(8, 10))
sns.barplot(data=top_roles, x='count', y='role', palette="viridis", orient='h')
plt.title("Top Person Roles (>10 occurrences)")
plt.tight_layout()

In [ ]:
top_roles

In [ ]:
# person_roles = [p['text'].lower() for r in records for p in r.get('persons', []) if p['label'] == 'Person_Role']
person_text_in_rel = [
    p['text'].lower()
    for rec in records
    for r in rec['relations']
    for p in rec.get('persons', [])
    if p['id'] in r.values() and p['label'].lower() == 'person_role'
]
person_roles_rel_df = pd.Series(person_text_in_rel).value_counts().rename_axis('role').reset_index(name='count')
top_roles = person_roles_rel_df[person_roles_rel_df['count'] > 10]

plt.figure(figsize=(8, 10))
sns.barplot(data=top_roles, x='count', y='role', palette="viridis", orient='h')
plt.title("Top Person Roles (>10 occurrences)")
plt.tight_layout()

In [ ]:
top_roles

In [ ]:
person_roles_rel_df['percentage'] = person_roles_rel_df['count'] / person_roles_rel_df['count'].sum() * 100
person_roles_rel_df.loc[person_roles_rel_df['role'].isin(['tenant', 'leaseholder'])].percentage.sum()

In [ ]:
distinct_linked_persons_per_note = [
    len(set(rel['to'] for rel in r.get('relations', [])))
    for r in records
]

linked_series = pd.Series(distinct_linked_persons_per_note)

pct_multi_linked = (linked_series > 1).mean() * 100
print(f"{pct_multi_linked:.1f}% of notes have needs linked to more than one distinct person")
linked_series.value_counts().sort_index()

### Label Co-occurence

In [ ]:
plot_cooccurrence_heatmap(records, taxonomy)


# Parent Co-occurence

In [ ]:
taxonomy_parent = taxonomy.replace({"cat_label": parent_mapping})
plot_cooccurrence_heatmap(records_parents, taxonomy_parent, show_annot=True)


## Analyse label distribution

In [ ]:
(irlbl_df, summary, ax) = analyze_category_distribution_from_dict(records, taxonomy, log_scale=False)

display(irlbl_df.style.background_gradient(cmap='Reds', subset=['IRLbl']))
display(summary)
plt.show()

 ## 4. Duplicate Handling & Agreement

 Group identical text contents, compute inter/intra-annotator agreement.

In [ ]:
grouped = defaultdict(list)
for record in records:
    grouped[record["text"]].append(record)

y_true_docs, y_pred_docs = [], []
all_labels = set()

for duplicates in grouped.values():
    if len(duplicates) > 1:
        duplicates.sort(key=lambda x: x['annotation_date'], reverse=True)
        new, old = duplicates[0], duplicates[1]
        
        old_spans = [(span["start"], span["end"], span["label"]) for span in old.get('needs', []) + old.get('persons', [])]
        new_spans = [(span["start"], span["end"], span["label"]) for span in new.get('needs', []) + new.get('persons', [])]

        y_true_docs.append(old_spans)
        y_pred_docs.append(new_spans)
        all_labels.update([s[2] for s in old_spans + new_spans])

if all_labels:
    evaluator = SpanEvaluator(all_labels=all_labels, logger=config.logger)
    results = evaluator.evaluate(y_true_docs, y_pred_docs)
    evaluator.print_report(results, title="INTRA-ANNOTATOR AGREEMENT REPORT")


In [ ]:
results['overall']['iou_0.5']

In [ ]:
with open(PROCESSED / 'gold_standard_gemini_pre_annotated.json', 'r', encoding='utf-8') as f:
    gem_data = json.load(f)
print(f"Total raw notes loaded: {len(gem_data)}")

grouped = defaultdict(list)
for record in gem_data:
    grouped[record['data']["note_content"]].append(record)

y_true_docs, y_pred_docs = [], []
all_labels = set()

for duplicates in grouped.values():
    if len(duplicates) > 1:
        # Assuming index 0 is newest, index 1 is oldest
        new, old = duplicates[0], duplicates[1]
        
        old_spans = [
            (span['value']["start"], span['value']["end"], label)
            for span in old.get('predictions', [{}])[0].get('result', [])
            if span.get('type') == 'labels'
            for label in span['value'].get("labels", [])
        ]
        new_spans = [
            (span['value']["start"], span['value']["end"], label)
            for span in new.get('predictions', [{}])[0].get('result', [])
            if span.get('type') == 'labels'
            for label in span['value'].get("labels", [])
        ]

        y_true_docs.append(old_spans)
        y_pred_docs.append(new_spans)
        
        all_labels.update([s[2] for s in old_spans + new_spans])

if all_labels:
    evaluator = SpanEvaluator(all_labels=all_labels, logger=config.logger)
    results = evaluator.evaluate(y_true_docs, y_pred_docs)
    evaluator.print_report(results, title="INTRA-ANNOTATOR AGREEMENT REPORT")

 ## 5. Class Imbalance Mitigation

 Merge low-prevalence labels to ensure models have enough training samples per class.
 The main targets for merging were examples with less than 40 examples.

In [ ]:
MERGE_MAP = {
    "care_care_experienced": "life_events_life_events",
    "cautions_dangerous_animals": "safety_risk_antisocial_behaviour",
    "mobility_service_need": "mobility_mobility_physical",
    "health_medical_life_sustaining": "health_terminally_ill",
    "reasonable_adjustments_mental_capacity": "reasonable_adjustments_communication_needs",
    "life_events_social_isolation": "life_events_life_events",
}

# Was not annotated consistently, so just confuses the model. Can be replaced with regexes easily.
CLASSES_TO_DROP = ["Person_Pronoun"] 

labels_to_drop = set(MERGE_MAP.keys())
labels_to_drop.update(CLASSES_TO_DROP)

# Combine labels in taxonomy
taxonomy_v3 = taxonomy.copy()
taxonomy_v3['values_hint'] = taxonomy_v3['values_hint'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

for old_label, new_label in MERGE_MAP.items():
    if old_label in taxonomy_v3['cat_label'].values and new_label in taxonomy_v3['cat_label'].values:
        old_idx = taxonomy_v3[taxonomy_v3['cat_label'] == old_label].index[0]
        new_idx = taxonomy_v3[taxonomy_v3['cat_label'] == new_label].index[0]
        
        taxonomy_v3.at[new_idx, 'regex'] = f"(?:{taxonomy_v3.at[old_idx, 'regex']})|(?:{taxonomy_v3.at[new_idx, 'regex']})"
        taxonomy_v3.at[new_idx, 'values_hint'] = taxonomy_v3.at[new_idx, 'values_hint'] + taxonomy_v3.at[old_idx, 'values_hint']

taxonomy_v3 = taxonomy_v3[~taxonomy_v3['cat_label'].isin(labels_to_drop)].reset_index(drop=True)

def filter_replace_annotations(ann):
    """Helper to filter out or merge labels."""
    if ann['label'] in labels_to_drop:
        if MERGE_MAP.get(ann['label']):
            ann['label'] = MERGE_MAP[ann['label']]
        else:
            return None # Drop the entity entirely.
    ann['label'] = ann['label'].lower() # Keep consistently lowercase.
    return ann 

# Combine labels in records.
records_v2 = []

for record in records:
    # Copy record except needs and persons
    new_record = copy.deepcopy(record)

    # Filter needs and persons, drop nones
    new_record["needs"] = [filter_replace_annotations(ann) for ann in record.get("needs", []) if filter_replace_annotations(ann)]
    new_record["persons"] = [filter_replace_annotations(ann) for ann in record.get("persons", []) if filter_replace_annotations(ann)]

    # Make sure all relation links to/from exist
    seen_ids = set()
    for ann in new_record["needs"] + new_record["persons"]:
        seen_ids.add(ann["id"])

    new_record["relations"] = [
        {"from": l["from"], "to": l["to"]}
        for l in record.get("relations", [])
        if l["from"] in seen_ids and l["to"] in seen_ids
    ]

    records_v2.append(new_record)


### Show new distribution

In [ ]:
(irlbl_df, summary, ax) = analyze_category_distribution_from_dict(records_v2, taxonomy_v3, log_scale=False)

display(irlbl_df.style.background_gradient(cmap='Reds', subset=['IRLbl']))
display(summary)
plt.show()

In [ ]:
label_counts = Counter()
for r in records_v2:
    labels_in_record = [l['label'] for l in r['needs'] + r['persons']]
    for label in labels_in_record:
        label_counts[label] += 1


freq = pd.Series(label_counts, name="frequency").sort_values(ascending=False)
max_freq = freq.max()

irlbl = max_freq / freq

irlbl_df = pd.DataFrame({
    "label": freq.index,
    "frequency": freq.values,
    "IRLbl": irlbl.values,
}).reset_index(drop=True)

summary_df = pd.DataFrame({
    "Metric": ["MeanIR", "MaxIR", "CVIR"],
    "Value": [irlbl.mean(), irlbl.max(), irlbl.std() / irlbl.mean()],
})
summary_df = summary_df.style.format({"Value": "{:.2f}"})

display(irlbl_df, summary_df)

## 7. Save as JSON

In [ ]:
# Dump all
save_json(config.output_path, records_v2, config.logger)